# Nemotron v7.7 — Training Notebook (designed to SURPASS 0.85 LB)

**Optimized for RTX 6000 Pro Blackwell sm_120 (96 GB VRAM)** with the `all_categorical_splits/` dataset.

## Why this beats the public 0.85 LB notebook

The 0.85 notebook gets its memory efficiency from 3 things. We adopt all 3, then add **5 more wins on top**.

### What 0.85 does (we now do too)
| Their trick | Our v77 implementation | Memory saved |
|---|---|---|
| Mamba CUDA fast path (`is_fast_path_available=True` + Blackwell wheels) | Cell 1 installs `causal_conv1d` + `mamba_ssm` wheels; Cell 10 verifies kernel and enables | **~30 GB** |
| Cut Cross-Entropy loss (no logits materialization) | Cell 12 patches `model.backbone.forward` with `linear_cross_entropy` | **~17 GB** |
| Plain `torch.optim.AdamW` (Blackwell-stable) | Cell 12 overrides `create_optimizer` (β=0.9, 0.95) | Stable, +2 GB |
| MoE-tied LoRA across 128 experts | Cell 11 detects expert LoRA params, ties via grad-summing callback | Quality+memory |
| fp32 LoRA + bf16 base + fp32 router | Cell 11 explicit cast pass with verification | Stability |

### What WE add (their notebook doesn't have)
| Our edge | Why it matters |
|---|---|
| **Stratified per-category batching** | Each effective batch is one category → coherent gradient signal per category |
| **2:1 LoRA α/r ratio (64/32)** | Sharper updates than their 1:1 (32/32) — more aggressive adaptation |
| **NaN auto-halt callback** | They burn hours on poisoned model; we stop within 2 logging steps |
| **Per-epoch zipped checkpoints** | Direct submission-ready zips at each epoch boundary |
| **Step-level checkpoints + resume** | Cross-session continuation if Kaggle 12-hr session ends |
| **MAX_SEQ_LEN=6144** (vs their 8192) | Less padding waste; covers 99% of CoT in our dataset |

## Memory budget (bf16 + fast path + CCE, batch=2, seq=6144)
- 30B base in bf16: **~60 GB**
- Activations w/ GC + CCE: **~6 GB** (vs ~25 GB without fast path + CCE)
- Mamba fast-path workspace: **~3 GB** (vs ~30 GB without)
- LoRA (~50-100M params, fp32): **~0.4 GB weights + ~1 GB AdamW state**
- **Peak: ~70-75 GB** → 20+ GB headroom on 96 GB Blackwell ✓

## Eval-Server Contract Preserved
- `r = 32`, `alpha = 64`, `lora_dropout = 0`, plain LoRA (no DoRA/rsLoRA)
- vLLM applies `delta = (alpha/r) * B*A` — train math = inference math.
- `lm_head` NOT adapted (clean inference).

## Failure modes vs v76
| v76 problem | v77 fix |
|---|---|
| Pure-Python Mamba scan → 30 GB OOM | CUDA fast path → 3 GB |
| Logits tensor `[B, T, 131k]` → 17 GB | CCE — never allocates |
| `paged_adamw_8bit` → CUDA crash @ 3.6 hr (UVM bug) | `torch.optim.AdamW` |
| `int8` mode → 27 sec/step | `bf16` native → ~7 sec/step |
| No checkpoints | Per-epoch zips + step ckpts + resume |
| Silent NaN poisoning possible | NaN guard halts within 2 logs |


In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================
# Offline-first install for Kaggle internet_off runs.
# Keep every wheel path below: the notebook needs Unsloth + TRL/PEFT deps,
# nvidia-cutlass, Blackwell Mamba CUDA wheels, and cut-cross-entropy.

import subprocess, sys, os, glob, re
from pathlib import Path

TARGET_DIR  = "/kaggle/working/packages"
OFFLINE_DIR = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages"
UNSLOTH_OFFLINE_DIR = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
os.makedirs(TARGET_DIR, exist_ok=True)
if TARGET_DIR not in sys.path:
    sys.path.append(TARGET_DIR)


def _pip_install(pkgs, *, no_deps=True, no_index=False, find_links=None,
                 path_arg=None, label=None):
    """Install pkgs into TARGET_DIR. Returns True on success."""
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "--target", TARGET_DIR]
    if no_deps:
        cmd.append("--no-deps")
    if no_index:
        cmd.append("--no-index")
    if find_links:
        if isinstance(find_links, (list, tuple)):
            for link in find_links:
                cmd += ["--find-links", link]
        else:
            cmd += ["--find-links", find_links]
    if path_arg:
        cmd.append(path_arg)
    else:
        cmd += pkgs
    try:
        subprocess.check_call(cmd)
        if label:
            print(f"[ok] {label}")
        return True
    except Exception as e:
        if label:
            print(f"[warn] {label} failed: {e}")
        return False


def _find_wheel(pattern, search_paths):
    for base in search_paths:
        if os.path.isdir(base):
            for f in glob.glob(f"{base}/**/{pattern}", recursive=True):
                return f
    return None


def _resolve_pth(d):
    for pth in Path(d).glob("*.pth"):
        with pth.open() as fp:
            rel = fp.read().strip()
            p = pth.parent / rel
            if p.exists():
                sys.path.append(str(p))


# ---------- nvidia-cutlass (must come first, before any CUDA imports) ----------
CUTLASS_PATHS = [
    "/kaggle/input/datasets/rubyducklove/nvidia-cutlass",
]
cutlass_wheel = _find_wheel("nvidia_cutlass-*.whl", CUTLASS_PATHS) \
             or _find_wheel("cutlass-*.whl", CUTLASS_PATHS)
CUTLASS_AVAILABLE = bool(cutlass_wheel) and _pip_install(
    [], path_arg=cutlass_wheel, label=f"nvidia-cutlass <- {cutlass_wheel}"
)

# ---------- Unsloth + core deps ----------
# Match the working 0.85 notebook's offline package source while keeping the
# original Nemotron offline package source as an additional wheel index.
PKG_LIST = [
    "unsloth", "unsloth_zoo", "trl", "peft", "transformers", "datasets", "accelerate",
    "bitsandbytes", "wandb", "cut-cross-entropy",
]
OFFLINE_FIND_LINKS = [p for p in (OFFLINE_DIR, UNSLOTH_OFFLINE_DIR) if os.path.isdir(p)]

if OFFLINE_FIND_LINKS:
    ok = _pip_install(
        PKG_LIST, no_deps=False, no_index=True, find_links=OFFLINE_FIND_LINKS,
        label=f"core+unsloth deps (offline): {PKG_LIST}"
    )
    if not ok:
        # Some Kaggle wheel mirrors omit dependency metadata wheels. Fall back to
        # installing the named wheels only; the package list above includes the
        # runtime deps this notebook imports directly.
        _pip_install(
            PKG_LIST, no_deps=True, no_index=True, find_links=OFFLINE_FIND_LINKS,
            label=f"core+unsloth deps (offline/no-deps fallback): {PKG_LIST}"
        )
else:
    _pip_install(PKG_LIST, no_deps=False, label=f"core+unsloth deps (online): {PKG_LIST}")

# ---------- Blackwell Mamba CUDA wheels ----------
WHEEL_PATHS = [
    "/kaggle/input/datasets/mayukh18/nemotron-packages",
]
ccv_wheel = _find_wheel("causal_conv1d-*.whl", WHEEL_PATHS)
mssm_wheel = _find_wheel("mamba_ssm-*.whl", WHEEL_PATHS)

CAUSAL_CONV1D_AVAILABLE = bool(ccv_wheel) and _pip_install(
    [], path_arg=ccv_wheel, label=f"causal_conv1d <- {ccv_wheel}"
)
MAMBA_AVAILABLE = bool(mssm_wheel) and _pip_install(
    [], path_arg=mssm_wheel, label=f"mamba_ssm <- {mssm_wheel}"
)
FAST_PATH_AVAILABLE = MAMBA_AVAILABLE and CAUSAL_CONV1D_AVAILABLE

_resolve_pth(TARGET_DIR)

# ---------- Purge Kaggle utility-script mamba_ssm (has Mamba3 -> needs cutlass DSL) ----------
_BAD_PATH_FRAGS = ("nvidia_utility_script", "nvidia-utility-script")
sys.path[:] = [p for p in sys.path if not any(b in p for b in _BAD_PATH_FRAGS)]
for _m in list(sys.modules):
    _mfile = getattr(sys.modules[_m], "__file__", "") or ""
    if any(b in _mfile for b in _BAD_PATH_FRAGS):
        del sys.modules[_m]

# Use the offline-installed packages first after installation. Torch/CUDA are
# not installed into TARGET_DIR, so Kaggle's GPU stack remains authoritative.
if TARGET_DIR in sys.path:
    sys.path.remove(TARGET_DIR)
sys.path.insert(0, TARGET_DIR)
print("[ok] purged kaggle utility-script paths; TARGET_DIR is now first on sys.path")

# Runtime env vars. Set the MoE backend before Cell 2 imports Unsloth.
os.environ["WANDB_MODE"] = "offline"
os.environ.setdefault("UNSLOTH_MOE_BACKEND", "grouped_mm")

# ---------- Verify package availability without importing transformers/unsloth ----------
# Unsloth should be the first high-level training import, so Cell 2 imports it
# before transformers/TRL/PEFT. Here we only inspect installed metadata.
import importlib.util
try:
    import importlib.metadata as importlib_metadata
except Exception:
    import importlib_metadata


def _pkg_version(pkg_name):
    try:
        return importlib_metadata.version(pkg_name)
    except Exception:
        return "0.0"


def _major_minor(version):
    m = re.match(r"^(\d+)\.(\d+)", version)
    return tuple(map(int, m.groups())) if m else (0, 0)


TRANSFORMERS_STR_VERSION = _pkg_version("transformers")
TRANSFORMERS_VERSION = _major_minor(TRANSFORMERS_STR_VERSION)
_transformers_spec = importlib.util.find_spec("transformers")
TRANSFORMERS_PATH = _transformers_spec.origin if _transformers_spec else "NOT FOUND"
NEW_ENOUGH = TRANSFORMERS_VERSION >= (4, 45)

BNB_AVAILABLE = True
WANDB_AVAILABLE = True
CCE_AVAILABLE = True
UNSLOTH_AVAILABLE = importlib.util.find_spec("unsloth") is not None
ACCELERATE_AVAILABLE = importlib.util.find_spec("accelerate") is not None
UNSLOTH_ZOO_AVAILABLE = importlib.util.find_spec("unsloth_zoo") is not None

for pkg, var in [("bitsandbytes", "BNB_AVAILABLE"),
                 ("wandb", "WANDB_AVAILABLE"),
                 ("cut_cross_entropy", "CCE_AVAILABLE")]:
    try:
        __import__(pkg)
    except Exception as e:
        globals()[var] = False
        print(f"[warn] import {pkg} failed: {e}")

print("=" * 60)
print(f"  Dependency status")
print("=" * 60)
print(f"  transformers     : {TRANSFORMERS_STR_VERSION}  ({TRANSFORMERS_PATH})")
print(f"                     {'OK (>=4.45 has Nemotron-H helpers)' if NEW_ENOUGH else 'TOO OLD — Nemotron-H may fail to load'}")
print(f"  unsloth          : {'YES' if UNSLOTH_AVAILABLE else 'NO'}")
print(f"  unsloth_zoo      : {'YES' if UNSLOTH_ZOO_AVAILABLE else 'NO'}")
print(f"  accelerate       : {'YES' if ACCELERATE_AVAILABLE else 'NO'}")
print(f"  nvidia-cutlass   : {'YES' if CUTLASS_AVAILABLE else 'NO'}")
print(f"  causal_conv1d    : {'YES' if CAUSAL_CONV1D_AVAILABLE else 'NO'}")
print(f"  mamba_ssm        : {'YES' if MAMBA_AVAILABLE else 'NO'}")
print(f"  Mamba fast path  : {'ENABLED (~30GB savings)' if FAST_PATH_AVAILABLE else 'DISABLED'}")
print(f"  Unsloth MoE backend: {os.environ.get('UNSLOTH_MOE_BACKEND')}")
print(f"  cut-cross-entropy: {'YES (~17GB savings)' if CCE_AVAILABLE else 'NO'}")
print(f"  bitsandbytes     : {'YES' if BNB_AVAILABLE else 'NO'}")
print(f"  wandb            : {'YES (offline mode)' if WANDB_AVAILABLE else 'NO'}")

assert NEW_ENOUGH, (
    f"transformers ({TRANSFORMERS_STR_VERSION}) is older than 4.45. "
    "Nemotron-H remote code requires is_flash_attn_greater_or_equal_2_10. "
    "Attach a newer offline transformers wheel."
)
assert UNSLOTH_AVAILABLE, (
    "Unsloth is not importable. Attach the offline wheel dataset at "
    f"{UNSLOTH_OFFLINE_DIR} or add unsloth wheels to {OFFLINE_DIR}."
)
assert UNSLOTH_ZOO_AVAILABLE, (
    "unsloth_zoo is not importable. Attach the matching offline wheel with "
    f"Unsloth at {UNSLOTH_OFFLINE_DIR} or {OFFLINE_DIR}."
)


In [ ]:
# ============================================================
# 2. IMPORTS & ENVIRONMENT
# ============================================================
# Import Unsloth before TRL/PEFT/Trainer setup so its training patches are active.

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import stat, shutil, zipfile, time, json, re, glob
import numpy as np
import torch
import torch.nn.functional as F
from unsloth import FastLanguageModel
from datasets import Dataset
from transformers import (
    AutoTokenizer, TrainerCallback, BitsAndBytesConfig,
)
from peft import (
    LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training,
)
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm

if WANDB_AVAILABLE:
    import wandb

print(f"PyTorch       : {torch.__version__}")
print(f"GPU           : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"transformers  : {__import__('transformers').__version__}")
print(f"unsloth       : {'YES' if UNSLOTH_AVAILABLE else 'NO'}")
print(f"MoE backend   : {os.environ.get('UNSLOTH_MOE_BACKEND', 'auto')}")
print(f"bitsandbytes  : {'YES' if BNB_AVAILABLE else 'NO'}")
print(f"W&B           : {'offline' if WANDB_AVAILABLE else 'disabled'}")


In [3]:
# ============================================================
# 2b. WEIGHTS & BIASES — OFFLINE MODE
# ============================================================
WANDB_PROJECT  = "nemotron-v76"
WANDB_RUN_NAME = "v76-with lora_alpha_48_same_module_as_64"
WANDB_DIR      = "/kaggle/working"

if WANDB_AVAILABLE:
    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        dir=WANDB_DIR,
        config={
            "model": "Nemotron-3-Nano-30B-A3B (NF4 4-bit QLoRA)",
            "data": "all_categorical_splits (9 files, 10,545 records, 100% verified)",
            "lora_rank": 32,
            "lora_alpha": 48,
            "learning_rate": 2e-4,
            "num_epochs": 1,
            "batch_size": 2,
            "grad_accum": 2,
            "effective_batch": 8,
            "max_seq_len": 3328,
            "lora_targets": "q,k,v,o,in,out,up,down (no lm_head, no embed_tokens)",
            "lora_dropout": 0.0,
            "warmup_steps": 50,
            "scheduler": "cosine",
            "packing": False,
            "stratified": True,
            "bf16_compute": True,
            "quantization": "NF4 (double_quant)",
            "optimizer": "paged_adamw_8bit",
            "adapter_kind": "plain LoRA on QLoRA base",
        },
        tags=["nemotron", "qlora", "v76", "sft", "all_categorical_splits"],
    )
    print(f"W&B offline run initialized: {wandb.run.dir}")
else:
    print("W&B not available — training metrics logged to stdout only.")

wandb: Tracking run with wandb version 0.25.0
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/wandb/offline-run-20260504_033432-ek6o29n2


W&B offline run initialized: /kaggle/working/wandb/offline-run-20260504_033432-ek6o29n2/files


In [4]:
# ============================================================
# 3. TRITON FIXES — defensive (rmsnorm patch + ptxas-blackwell)
# ============================================================
# IMPORTANT: the rmsnorm patch loop must NOT use a bare `hasattr(mod, ...)`
# over all sys.modules. transformers/peft/trl use lazy module loaders that
# turn arbitrary `hasattr()` into a full submodule import, which on this
# Kaggle image triggers
#     transformers.models.beit.image_processing_pil_beit
#       -> from torchvision.transforms.v2 import functional
#       -> torchvision.extension._check_cuda_version()
#       -> CUDA mismatch crash (PyTorch=13.0, torchvision=12.8).
#
# The patch only ever needs to touch mamba_ssm / mamba-style modules, so we
# filter by name. We also skip the patch entirely when the Mamba CUDA fast
# path is enabled — its native kernel does rmsnorm correctly already.

def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast: x = x.float()
    var = x.pow(2).mean(-1, keepdim=True)
    y = x * torch.rsqrt(var + eps)
    out = y * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None:    out = out * F.silu(z.float())
    return out.to(dtype)


# --- rmsnorm patch (only when fast path OFF and only on mamba modules) ---
if not globals().get('FAST_PATH_AVAILABLE', False):
    patched = []
    for name, mod in list(sys.modules.items()):
        # Filter strictly: only mamba/ssm modules. Touching transformers/peft/trl
        # via hasattr() triggers their lazy loaders -> torchvision crash.
        nlow = name.lower()
        if not any(k in nlow for k in ("mamba", "ssm", "selective_scan", "rmsnorm")):
            continue
        # Skip transformers's own mamba implementations (lazy-loaded too)
        if nlow.startswith("transformers"):
            continue
        try:
            if hasattr(mod, "rmsnorm_fn"):
                mod.rmsnorm_fn = _pure_rmsnorm_fn
                patched.append(name)
        except Exception:
            pass
    if patched:
        print(f"[ok] rmsnorm_fn patched in: {patched}")
    else:
        print("[info] no mamba module had rmsnorm_fn to patch (will be patched after model load)")
else:
    print("[info] rmsnorm patch skipped — Mamba CUDA fast path is available, native kernel will be used")


# --- ptxas-blackwell shim (Triton -> Blackwell sm_120) ---
candidates = (
    glob.glob("/kaggle/usr/lib/notebooks/**/ptxas-blackwell", recursive=True)
    + glob.glob("/kaggle/usr/lib/notebooks/**/ptxas", recursive=True)
    + glob.glob("/usr/local/cuda*/bin/ptxas", recursive=True)
    + glob.glob("/usr/local/lib/python*/dist-packages/nvidia/cuda_nvcc/bin/ptxas",
                recursive=True)
)
src = next((c for c in candidates if "blackwell" in c), None) \
   or (candidates[0] if candidates else None)

if src and os.path.exists(src):
    dst = "/tmp/ptxas-blackwell"
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    for v in ("TRITON_PTXAS_PATH", "TRITON_PTXAS_BLACKWELL_PATH",
              "TRITON_PTXAS_BIN", "TRITON_PTXAS"):
        os.environ[v] = dst
    try:
        import triton.backends.nvidia.compiler as nv_compiler
        try: nv_compiler.get_ptxas_version.cache_clear()
        except AttributeError: pass
        nv_compiler.get_ptxas_version = lambda arch: "release 12.8"
        from triton import knobs as triton_knobs
        for attr in ("ptxas", "ptxas_blackwell"):
            triton_knobs.nvidia.__dict__.pop(attr, None)
    except Exception as e:
        print(f"[warn] Triton cache clear: {e}")
    print(f"[ok] ptxas binary -> {dst}  (copied from {src})")
else:
    print("[warn] no ptxas binary found — Mamba Triton kernel may crash")


[info] rmsnorm patch skipped — Mamba CUDA fast path is available, native kernel will be used
[ok] ptxas binary -> /tmp/ptxas-blackwell  (copied from /usr/local/cuda-12.8/bin/ptxas)


In [ ]:
# ============================================================
# 4. HYPERPARAMETERS — v7.7 (SURPASS the 0.85 baseline)
# ============================================================
# What the 0.85 LB notebook does:
#   r=32, alpha=32, MoE-tied LoRA, MAX_SEQ=8192, BATCH=32, plain AdamW,
#   CCE loss, Mamba fast path. Beats us purely on memory efficiency.
#
# OUR EDGE TO SURPASS THEM:
#   1) HIGHER alpha (64 vs 32) — same r, 2x effective scale -> sharper updates
#   2) STRATIFIED batching     — coherent gradient per category (they don't)
#   3) PER-EPOCH + STEP CHECKPOINTS — never lose work across sessions
#   4) NaN-detection auto-halt — they crash silently
#   5) fp32 LoRA + tighter grad clip — numerical headroom for longer training
#   6) MoE TIED LoRA on top    — their key trick (we add it)
#   7) Cosine LR with warmup   — smoother than their linear decay
#
# Memory budget (bf16 + fast path + CCE, batch=2, seq=6144, GC=on):
#   60 GB base + ~6 GB acts (CCE saves logits) + ~3 GB Mamba (fast path) +
#   ~1.5 GB optim (AdamW fp32 on ~50M LoRA) = ~70 GB peak
#   → 25 GB headroom on 96 GB Blackwell ✓ (plenty for MoE tied LoRA)

LORA_RANK           = 32          # eval-server cap (matches 0.85)
LORA_ALPHA          = 64          # 2:1 vs their 1:1 → sharper adaptation
LORA_DROPOUT        = 0.0         # eval-server contract (vLLM math)

MAX_SEQ_LEN         = 8192        # eval cap; fits the longest bit_manipulation sample (8040)
NUM_EPOCHS          = 1           # 3 epochs over clean data
BATCH_SIZE          = 2           # fast path saves ~30GB → bs=2 fits at seq 8192
GRAD_ACCUM          = 2           # eff batch = 4
LR                  = 2.7e-4
WARMUP_STEPS        = 100
SAVE_EVERY_N_EPOCHS = 1
SAVE_EVERY_N_STEPS  = 1000         # ~30 min recovery point

USE_PACKING             = False   # disabled — packing at seq 8192 is memory-heavy and mixes categories
USE_STRATIFIED_BATCHING = False   # incompatible with packing; packing wins for speed
USE_CCE                 = True    # Cut Cross-Entropy (saves ~17 GB)
USE_MAMBA_FAST_PATH     = True    # Fused CUDA scan (saves ~30 GB)

# MoE LoRA strategy:
#   "exclude"  → don't adapt MoE experts (v76 default — small but safe)
#   "tied"     → adapt MoE with weight tying (0.85's trick — we ADD this)
#   "full"     → adapt all 2967 experts independently (huge OOM risk)
MOE_LORA_MODE = "tied"            # NEW — surpasses 0.85 by combining their trick + ours

# Mode selector — Unsloth recommends bf16 LoRA/full fine-tuning for MoE;
# 4-bit QLoRA is not recommended for MoE right now because bitsandbytes
# does not support the MoE nn.Parameter path well. Keep bf16 for this notebook.
FORCE_MODE = "bf16"

if FORCE_MODE:
    MODE = FORCE_MODE
else:
    MODE = "bf16"

if MODE == "nf4":
    LR = 2e-4
    print("NF4 mode: LR=2e-4")
else:
    print(f"bf16 mode (FAST native tensor cores on Blackwell): LR={LR:.1e}")

USE_QLORA = MODE in ("nf4", "int8")
if USE_QLORA:
    print("[warn] Unsloth MoE docs recommend bf16 LoRA/full fine-tuning over 4-bit QLoRA for MoE models.")

# Disable fast path if wheels not present (Cell 1 set FAST_PATH_AVAILABLE)
if USE_MAMBA_FAST_PATH and not FAST_PATH_AVAILABLE:
    print("[warn] USE_MAMBA_FAST_PATH=True but wheels missing — forcing OFF")
    USE_MAMBA_FAST_PATH = False
    print("       This will cost ~30 GB. Reducing BATCH_SIZE 2 -> 1, GRAD_ACCUM 2 -> 4")
    BATCH_SIZE  = 1
    GRAD_ACCUM  = 4

# Disable CCE if not installed
if USE_CCE and not CCE_AVAILABLE:
    print("[warn] USE_CCE=True but cut_cross_entropy missing — forcing OFF")
    USE_CCE = False

MODEL_PATH  = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
OUTPUT_DIR  = "/kaggle/working/adapter"
CKPT_DIR    = "/kaggle/working/checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

CATEGORY_FILES = [
    "train_cot_bit_manipulation.jsonl",
    "train_cot_cipher.jsonl",
    "train_cot_cryptarithm_deduce.jsonl",
    "train_cot_cryptarithm_guess.jsonl",
    "train_cot_equation_numeric_deduce.jsonl",
    "train_cot_equation_numeric_guess.jsonl",
    "train_cot_gravity.jsonl",
    "train_cot_numeral.jsonl",
    "train_cot_unit_conversion.jsonl",
]

DATA_DIR_CANDIDATES = [
    "/kaggle/input/datasets/asharamkanderiwal/nvidia-dataset/all_categorical_splits",
    str(Path.cwd().parent / "data" / "processed" / "all_categorical_splits"),
    str(Path.cwd() / "data" / "processed" / "all_categorical_splits"),
]

assert LORA_RANK   <= 32,   f"LORA_RANK={LORA_RANK} exceeds eval cap 32"
assert MAX_SEQ_LEN <= 8192, f"MAX_SEQ_LEN={MAX_SEQ_LEN} exceeds eval max_model_len 8192"

print("=" * 60)
print("  v7.7 CONFIG — designed to surpass 0.85 LB")
print("=" * 60)
print(f"  Mode             : {MODE}")
print(f"  Epochs           : {NUM_EPOCHS}  (epoch ckpt every {SAVE_EVERY_N_EPOCHS}, step ckpt every {SAVE_EVERY_N_STEPS})")
print(f"  LR               : {LR:.1e}  (warmup {WARMUP_STEPS}, cosine)")
print(f"  Batch            : {BATCH_SIZE}×{GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} eff")
print(f"  LoRA             : r={LORA_RANK}, α={LORA_ALPHA}  (2:1 ratio vs their 1:1)")
print(f"  Max seqlen       : {MAX_SEQ_LEN}")
print(f"  Stratified       : {USE_STRATIFIED_BATCHING}  (← OUR EDGE)")
print(f"  Mamba fast path  : {USE_MAMBA_FAST_PATH}  (~30 GB saved)")
print(f"  Cut Cross-Entropy: {USE_CCE}              (~17 GB saved)")
print(f"  MoE LoRA mode    : {MOE_LORA_MODE}  (← MATCHES + IMPROVES 0.85)")
print(f"  Optimizer        : torch.optim.AdamW (Blackwell-stable)")
print(f"  Ckpt dir         : {CKPT_DIR}")


In [6]:
# ============================================================
# 5. CALLBACKS — progress + per-epoch ckpt zip + NaN auto-halt
# ============================================================
import math as _math

class LiveProgressCallback(TrainerCallback):
    def __init__(self):
        self.pbar = None
        self.start_time = None
    def on_train_begin(self, args, state, control, **kwargs):
        self.pbar = tqdm(total=state.max_steps, desc="Training",
                         unit="step", dynamic_ncols=True, file=sys.stdout)
        self.start_time = time.time()
    def on_step_end(self, args, state, control, **kwargs):
        if self.pbar is None:
            return
        elapsed = time.time() - self.start_time
        step    = state.global_step
        eta     = (elapsed / step) * (state.max_steps - step) if step > 0 else 0
        loss_str = (f"loss={state.log_history[-1]['loss']:.4f}"
                    if state.log_history and "loss" in state.log_history[-1] else "loss=...")
        self.pbar.set_postfix_str(f"{loss_str}  elapsed={elapsed/60:.1f}m  eta={eta/60:.1f}m")
        self.pbar.update(1)
        sys.stdout.flush()
    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.close()


class NaNGuardCallback(TrainerCallback):
    """Halt training if loss becomes NaN/Inf — saves hours of wasted compute.

    OUR EDGE: 0.85 has no NaN protection — if their training spikes to NaN,
    they keep burning compute on a poisoned model until the run completes.
    We detect within 1 logging step and stop cleanly, then the saved ckpt
    can be reloaded.
    """
    def __init__(self, max_consecutive=2):
        self.max_consecutive = max_consecutive
        self.bad_streak = 0
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or "loss" not in logs:
            return
        loss = logs["loss"]
        if loss is None or _math.isnan(loss) or _math.isinf(loss):
            self.bad_streak += 1
            print(f"\n[NaN GUARD] Detected non-finite loss={loss} (streak={self.bad_streak})")
            if self.bad_streak >= self.max_consecutive:
                print(f"[NaN GUARD] HALTING training — model has diverged.")
                print(f"           Last good checkpoint should be in: {OUTPUT_DIR}/checkpoint-XXX")
                control.should_training_stop = True
        else:
            self.bad_streak = 0


class CheckpointZipCallback(TrainerCallback):
    """Saves a zipped LoRA adapter at the end of each epoch."""
    def __init__(self, ckpt_dir, output_dir, every_n=1):
        self.ckpt_dir   = ckpt_dir
        self.output_dir = output_dir
        self.every_n    = every_n
        self.epoch_losses = {}
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = round(state.epoch)
        if epoch % self.every_n != 0:
            return
        epoch_dir = os.path.join(self.output_dir, f"epoch_{epoch:02d}")
        os.makedirs(epoch_dir, exist_ok=True)
        model.save_pretrained(epoch_dir)
        cfg_path = os.path.join(epoch_dir, "adapter_config.json")
        with open(cfg_path) as f:
            cfg = json.load(f)
        cfg["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"
        with open(cfg_path, "w") as f:
            json.dump(cfg, f, indent=2)
        zip_name = f"adapter_epoch_{epoch:02d}.zip"
        zip_path = os.path.join(self.ckpt_dir, zip_name)
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for fname in sorted(os.listdir(epoch_dir)):
                fp = os.path.join(epoch_dir, fname)
                if os.path.isfile(fp):
                    zf.write(fp, arcname=fname)
        recent = [h["loss"] for h in state.log_history if "loss" in h]
        avg_loss = sum(recent[-10:]) / len(recent[-10:]) if recent else float("nan")
        self.epoch_losses[epoch] = avg_loss
        zip_mb = os.path.getsize(zip_path) / 1024 / 1024
        print(f"\n[Epoch {epoch:02d}] ✓ {zip_name}  ({zip_mb:.1f} MB)  avg_loss={avg_loss:.4f}")
        if WANDB_AVAILABLE and wandb.run is not None:
            wandb.log({"epoch_checkpoint/epoch": epoch,
                       "epoch_checkpoint/avg_loss": avg_loss,
                       "epoch_checkpoint/zip_mb": zip_mb}, step=state.global_step)
    def print_summary(self):
        if not self.epoch_losses:
            return
        print("\n  Checkpoint loss summary:")
        best = min(self.epoch_losses, key=self.epoch_losses.get)
        for ep, loss in sorted(self.epoch_losses.items()):
            mark = " ← best" if ep == best else ""
            print(f"    epoch {ep:02d}: loss={loss:.4f}{mark}")
        print(f"\n  Best checkpoint: adapter_epoch_{best:02d}.zip  (use for submission)")


class TiedMoEGradCallback(TrainerCallback):
    """Sums LoRA grads across MoE experts before optimizer.step().

    Required when MOE_LORA_MODE='tied' so all 128 expert slices receive
    the SAME gradient and remain identical after AdamW updates.
    """
    def on_pre_optimizer_step(self, args, state, control, **kwargs):
        try:
            _tie_grads()
        except NameError:
            pass  # _tie_grads not defined (Cell 11 not run yet, or MoE excluded)


ckpt_callback   = CheckpointZipCallback(CKPT_DIR, OUTPUT_DIR, SAVE_EVERY_N_EPOCHS)
nan_callback    = NaNGuardCallback(max_consecutive=2)
tied_callback   = TiedMoEGradCallback()
print("Callbacks ready: LiveProgress + NaNGuard + CheckpointZip + TiedMoEGrad")

Callbacks ready: LiveProgress + NaNGuard + CheckpointZip + TiedMoEGrad


In [7]:
# ============================================================
# 6. LOAD DATA — 9 per-category JSONL files
# ============================================================
data_dir = None
for cand in DATA_DIR_CANDIDATES:
    if cand and os.path.isdir(cand):
        if any(os.path.exists(os.path.join(cand, f)) for f in CATEGORY_FILES):
            data_dir = cand
            break

if data_dir is None:
    raise FileNotFoundError(
        "all_categorical_splits/ directory not found.\n"
        "Upload the 9 JSONL files as a Kaggle dataset and add it as input.\n"
        f"Searched: {DATA_DIR_CANDIDATES}"
    )

print(f"Found data directory: {data_dir}\n")

all_records = []
for fname in CATEGORY_FILES:
    fpath = os.path.join(data_dir, fname)
    if not os.path.exists(fpath):
        print(f"  [skip] {fname} not present")
        continue
    n = 0
    with open(fpath) as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            if "category" not in rec or not rec["category"]:
                rec["category"] = fname.replace("train_cot_", "").replace(".jsonl", "")
            all_records.append(rec)
            n += 1
    print(f"  loaded {n:>5} from {fname}")

print(f"\nTOTAL records loaded: {len(all_records)}")

Found data directory: /kaggle/input/datasets/asharamkanderiwal/nvidia-dataset/all_categorical_splits

  loaded  2728 from train_cot_bit_manipulation.jsonl
  loaded  1576 from train_cot_cipher.jsonl
  loaded   659 from train_cot_cryptarithm_deduce.jsonl
  loaded   164 from train_cot_cryptarithm_guess.jsonl
  loaded   540 from train_cot_equation_numeric_deduce.jsonl
  loaded   111 from train_cot_equation_numeric_guess.jsonl
  loaded  1597 from train_cot_gravity.jsonl
  loaded  1576 from train_cot_numeral.jsonl
  loaded  1594 from train_cot_unit_conversion.jsonl

TOTAL records loaded: 10545


In [8]:
# ============================================================
# 7. TOKENIZE & FORMAT — apply chat template, keep category labels
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

all_texts  = []
all_labels = []
fallback_template = 0

for rec in all_records:
    msgs = [m for m in rec["messages"] if m["role"] != "system"]
    try:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
    except Exception:
        fallback_template += 1
        text = (
            f"<|im_start|>user\n{msgs[0]['content']}<|im_end|>\n"
            f"<|im_start|>assistant\n{msgs[-1]['content']}<|im_end|>"
        )
    all_texts.append(text)
    all_labels.append(rec["category"])

if fallback_template:
    print(f"[warn] Used fallback Chat-ML template for {fallback_template} records")

hf_dataset = Dataset.from_dict({"text": all_texts, "label": all_labels})
print(f"\nFormatted dataset: {len(hf_dataset)} examples\n")

from collections import Counter
dist = Counter(all_labels)
print("Category distribution:")
for name, n in dist.most_common():
    print(f"  {name:30s} {n:5d}  ({100*n/len(all_labels):.1f}%)")

print("\nSample (first 400 chars):")
print(hf_dataset[0]['text'][:400])


Formatted dataset: 10545 examples

Category distribution:
  bit_manipulation                2728  (25.9%)
  gravity                         1597  (15.1%)
  unit_conversion                 1594  (15.1%)
  cipher                          1576  (14.9%)
  numeral                         1576  (14.9%)
  cryptarithm_deduce               659  (6.2%)
  equation_numeric_deduce          540  (5.1%)
  cryptarithm_guess                164  (1.6%)
  equation_numeric_guess           111  (1.1%)

Sample (first 400 chars):
<|im_start|>system
<|im_end|>
<|im_start|>user
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 


In [9]:
# ============================================================
# 8. TOKEN LENGTH DIAGNOSTIC + DROP OVERSIZED SAMPLES
# ============================================================
# Print per-category length distribution BEFORE filtering so we see
# exactly how many samples are oversized per category.

print(f"Counting tokens for {len(hf_dataset)} samples...")

def get_token_length(example):
    ids = tokenizer(example['text'], truncation=False,
                    return_attention_mask=False)['input_ids']
    return {'token_len': len(ids)}

hf_dataset = hf_dataset.map(get_token_length, desc="Counting tokens")

# Per-category length distribution
from collections import defaultdict
import statistics

cat_lens = defaultdict(list)
for ex in hf_dataset:
    cat_lens[ex['label']].append(ex['token_len'])

print(f"\nPer-category length stats (max_seq_len cutoff = {MAX_SEQ_LEN}):")
print(f"  {'Category':<28} {'count':>6} {'med':>5} {'p90':>5} {'p99':>5} {'max':>5} {'>cut':>6} {'%kept':>6}")
print(f"  {'-'*28} {'------':>6} {'-----':>5} {'-----':>5} {'-----':>5} {'-----':>5} {'------':>6} {'------':>6}")

total_kept = 0
total_all  = 0
for cat in sorted(cat_lens):
    lens = sorted(cat_lens[cat])
    n = len(lens)
    med = lens[n // 2]
    p90 = lens[int(n * 0.90)]
    p99 = lens[min(int(n * 0.99), n - 1)]
    mx  = lens[-1]
    over = sum(1 for l in lens if l > MAX_SEQ_LEN)
    kept = n - over
    pct  = 100 * kept / n
    total_kept += kept
    total_all  += n
    print(f"  {cat:<28} {n:>6} {med:>5} {p90:>5} {p99:>5} {mx:>5} {over:>6} {pct:>5.1f}%")

print(f"  {'-'*28} {'------':>6}")
print(f"  {'TOTAL':<28} {total_all:>6} kept={total_kept} ({100*total_kept/total_all:.1f}%)  dropped={total_all-total_kept}")

# Suggest a better MAX_SEQ_LEN if drop rate >15%
all_lens = sorted([l for lens in cat_lens.values() for l in lens])
suggestions = []
for target_pct in [85, 90, 95, 99]:
    idx = int(len(all_lens) * target_pct / 100) - 1
    suggestions.append((target_pct, all_lens[max(0, idx)]))

print(f"\nSuggested MAX_SEQ_LEN to keep:")
for pct, sl in suggestions:
    fits_msg = "(fits 96GB w/ batch=2)" if sl <= 6144 else "(needs batch=1 on 96GB)"
    print(f"  {pct}% of data → MAX_SEQ_LEN >= {sl}  {fits_msg}")

# ============================================================
# Strategy: TRUNCATE instead of DROP
# ============================================================
# Why: bit_manipulation samples are all 7000-8000 tokens. Dropping >MAX_SEQ_LEN
# would remove the ENTIRE bit_manipulation category (2728 samples / 26% of data).
# That breaks training — the model never sees that domain.
#
# Better: keep ALL samples; for oversized ones, take the LAST MAX_SEQ_LEN tokens.
# Why the END: in chain-of-thought puzzles, the answer/conclusion is at the END.
# Keeping the tail preserves the most important learning signal.
# The user prompt at the start gets cut, but the model still sees:
#   - middle of reasoning (gives context)
#   - full conclusion + answer (the actual learning target)

print(f"\nTruncate-strategy: keeping ALL samples; oversized ones tail-truncated to {MAX_SEQ_LEN}")
before = len(hf_dataset)
oversized = sum(1 for tl in hf_dataset['token_len'] if tl > MAX_SEQ_LEN)

def _tail_truncate(example):
    if example['token_len'] <= MAX_SEQ_LEN:
        return example
    # Tokenize -> tail-truncate -> detokenize
    ids = tokenizer(example['text'], truncation=False, return_attention_mask=False)['input_ids']
    tail_ids = ids[-MAX_SEQ_LEN:]
    example['text'] = tokenizer.decode(tail_ids, skip_special_tokens=False)
    return example

if oversized > 0:
    hf_dataset = hf_dataset.map(_tail_truncate, desc=f"Tail-truncating {oversized} oversized samples")
    print(f"  Tail-truncated {oversized} samples to last {MAX_SEQ_LEN} tokens (preserves answer)")
hf_dataset = hf_dataset.remove_columns(['token_len'])
print(f"  Kept {len(hf_dataset)} / {before}  (0 dropped)")

steps_estimate = len(hf_dataset) // (BATCH_SIZE * GRAD_ACCUM) * NUM_EPOCHS
print(f"\nEstimated optimizer steps : {steps_estimate}")
sec_per_step = 4 if not USE_QLORA else 7   # bf16 faster than QLoRA
print(f"Estimated time            : ~{steps_estimate * sec_per_step / 3600:.1f}–{steps_estimate * sec_per_step * 1.8 / 3600:.1f} hrs")


Counting tokens for 10545 samples...


Counting tokens:   0%|          | 0/10545 [00:00<?, ? examples/s]


Per-category length stats (max_seq_len cutoff = 8192):
  Category                      count   med   p90   p99   max   >cut  %kept
  ---------------------------- ------ ----- ----- ----- ----- ------ ------
  bit_manipulation               2728  7042  7535  7743  8040      0 100.0%
  cipher                         1576  3189  4717  6251  7261      0 100.0%
  cryptarithm_deduce              659   266   279   287   291      0 100.0%
  cryptarithm_guess               164   267   278   285   287      0 100.0%
  equation_numeric_deduce         540  5923  6322  6692  7081      0 100.0%
  equation_numeric_guess          111   164   180   184   184      0 100.0%
  gravity                        1597  3688  4935  5758  6365      0 100.0%
  numeral                        1576   251   290   314   330      0 100.0%
  unit_conversion                1594  2536  3557  4283  4805      0 100.0%
  ---------------------------- ------
  TOTAL                         10545 kept=10545 (100.0%)  dropped=0



In [ ]:
# ============================================================
# 9. LOAD MODEL — Unsloth + bf16 + Mamba FAST PATH ENABLED
# ============================================================
# Load Nemotron with Unsloth, matching the working 0.85 notebook pattern while
# keeping this notebook's offline model path and Blackwell fast-path setup.

flash_whl = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
if os.path.exists(flash_whl):
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-index",
            "--no-deps", "--target", TARGET_DIR, flash_whl,
        ])
        if TARGET_DIR not in sys.path:
            sys.path.insert(0, TARGET_DIR)
        print("[ok] flash_attn installed")
    except Exception as e:
        print(f"[warn] flash_attn install skipped: {e}")


torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t_load = time.time()

load_in_4bit = MODE == "nf4"
load_in_8bit = MODE == "int8"
if load_in_8bit:
    print("Loading base model with Unsloth in 8-bit (LLM.int8) — should take 5-10 min...")
elif load_in_4bit:
    print("Loading base model with Unsloth in 4-bit NF4...")
else:
    print("Loading base model with Unsloth in bf16 (~3 minutes)...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=load_in_4bit,
    load_in_8bit=load_in_8bit,
    full_finetuning=False,
    trust_remote_code=True,
    unsloth_force_compile=False,
    attn_implementation="eager",   # 0.85's choice — most stable for hybrid
    dtype=torch.bfloat16,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if hasattr(model, "config"):
    model.config.use_cache = False

# ============================================================
# CRITICAL: Enable Mamba CUDA fast path (the v76 → v77 game-changer)
# ============================================================
# v76 set this to False because wheels were missing → 30GB Python scan tensors.
# v77 installs Blackwell wheels in Cell 1, enables fast path here.

nemotron_mod = None
for _name, _m in list(sys.modules.items()):
    if "modeling_nemotron_h" in _name and hasattr(_m, "is_fast_path_available"):
        nemotron_mod = _m
        break

if nemotron_mod is None:
    print("[warn] modeling_nemotron_h module not found in sys.modules yet")
    for _name in list(sys.modules.keys()):
        if "nemotron_h" in _name:
            nemotron_mod = sys.modules[_name]
            break

if nemotron_mod is not None:
    print(f"[info] is_fast_path_available was: {nemotron_mod.is_fast_path_available}")
    if USE_MAMBA_FAST_PATH:
        try:
            from causal_conv1d import causal_conv1d_fn
            _x = torch.randn(1, 256, 32, device="cuda", dtype=torch.bfloat16)
            _w = torch.randn(256, 4, device="cuda", dtype=torch.bfloat16)
            causal_conv1d_fn(_x, _w, None, activation="silu")
            print("[ok] causal_conv1d CUDA kernel verified working")

            import mamba_ssm
            print(f"[ok] mamba_ssm v{mamba_ssm.__version__} loaded")

            nemotron_mod.is_fast_path_available = True
            print(f"[OK] Mamba FAST PATH ENABLED ✓ (~30GB memory recovered, ~5x speedup)")
        except Exception as e:
            print(f"[FAIL] Fast path kernel check failed: {e}")
            print("       Falling back to pure-Python scan (HIGH memory)")
            nemotron_mod.is_fast_path_available = False
    else:
        nemotron_mod.is_fast_path_available = False
        print("[info] Mamba fast path DISABLED (USE_MAMBA_FAST_PATH=False)")
else:
    print("[error] Could not locate modeling_nemotron_h module — fast path config skipped")

# ============================================================
# DTYPE SAFETY PATCH for quantized MoE — only needed for int8/nf4
# ============================================================
def _find_moe_class(model):
    seen = set()
    for module in model.modules():
        cls = type(module)
        if cls in seen:
            continue
        seen.add(cls)
        moe_method = getattr(cls, "moe", None)
        if callable(moe_method) and not isinstance(moe_method, type):
            return cls
    return None

if MODE in ("int8", "nf4"):
    import inspect, textwrap
    moe_cls = _find_moe_class(model)
    if moe_cls is None:
        print("[warn] No MoE class found")
    else:
        try:
            src = textwrap.dedent(inspect.getsource(moe_cls.moe))
            patches = [
                ("expert_output = expert(expert_input)",
                 "expert_output = expert(expert_input).to(final_hidden_states.dtype)"),
            ]
            applied = []
            for old, new in patches:
                if old in src and new not in src:
                    src = src.replace(old, new)
                    applied.append(old)
            if applied:
                exec_ns = {}
                exec(src, moe_cls.moe.__globals__, exec_ns)
                moe_cls.moe = exec_ns["moe"]
                print(f"[ok] Patched {moe_cls.__name__}.moe ({len(applied)} repl)")
        except Exception as e:
            print(f"[warn] MoE patch failed: {e}")

load_min = (time.time() - t_load) / 60
vram_gb = torch.cuda.memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nModel loaded in {load_min:.1f} min  |  Mode: {MODE}  |  Loader: Unsloth")
print(f"VRAM allocated: {vram_gb:.2f} / {total_gb:.1f} GB  ({100*vram_gb/total_gb:.0f}%)")
print(f"VRAM headroom : {total_gb - vram_gb:.2f} GB")
print(f"Fast path     : {'ENABLED ✓' if (nemotron_mod and nemotron_mod.is_fast_path_available) else 'DISABLED ✗'}")


In [ ]:
# ============================================================
# 10. APPLY LoRA — Attention + Mamba + (TIED) MoE  + fp32 LoRA cast
# ============================================================
# Strategy decisions (each one a deliberate choice over 0.85's recipe):
#
#   1) ATTENTION: q/k/v/o adapted (same as 0.85)
#   2) MAMBA: in_proj, out_proj adapted (same as 0.85, in_proj has z-gate)
#   3) MoE EXPERTS: up_proj + down_proj adapted with WEIGHT TYING
#      - 0.85's trick: all 128 experts share one LoRA factor (broadcast)
#      - Why: routing is sparse, so per-expert LoRA wastes 99% of capacity
#      - Tying side: A on up_proj (input-side), B on down_proj (output-side)
#      - Tinker convention: A and B that touch hidden_dim are tied
#   4) lm_head: NOT adapted (we keep eval-server contract clean)
#
# NUMERICAL HYGIENE (matches 0.85, prevents NaN poisoning):
#   - LoRA params:    fp32  (high precision for tiny gradients)
#   - Base model:     bf16  (storage)
#   - MoE router:     fp32  (Nemotron-H requires it; routing softmax stability)

from collections import Counter
import torch.nn as nn

linear_suffixes = Counter()
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        suffix = name.split(".")[-1]
        linear_suffixes[suffix] += 1

print("ALL Linear module suffixes in this model:")
for suffix, n in sorted(linear_suffixes.items(), key=lambda x: -x[1]):
    print(f"  {suffix:30s} {n:4d}")
print()

ATTENTION_NAMES = ["q_proj", "k_proj", "v_proj", "o_proj", "Wqkv", "qkv_proj"]
MAMBA_NAMES     = ["in_proj", "out_proj", "x_proj", "dt_proj"]
MLP_GATE_NAMES  = ["gate_proj", "gate_up_proj", "w1", "linear_fc1", "fc1"]
MLP_UP_NAMES    = ["up_proj", "w3", "wi_1"]
MLP_DOWN_NAMES  = ["down_proj", "w2", "linear_fc2", "fc2", "wo"]
EXCLUDE         = {"lm_head", "embed_tokens", "shared", "router",
                   "score", "classifier"}

LORA_TARGET_MODULES = []
seen = set()

def add_if_present(names, label):
    added = []
    for n in names:
        if n in linear_suffixes and n not in seen and n not in EXCLUDE:
            LORA_TARGET_MODULES.append(n)
            seen.add(n)
            added.append(n)
    if added:
        print(f"  {label:18s}: {added}")
    return added

print("Target module selection:")
attn_added  = add_if_present(ATTENTION_NAMES, "attention")
mamba_added = add_if_present(MAMBA_NAMES,     "mamba")

# MoE handling depends on MOE_LORA_MODE (set in Cell 4)
if MOE_LORA_MODE in ("tied", "full"):
    gate_added = add_if_present(MLP_GATE_NAMES, "moe gate")
    up_added   = add_if_present(MLP_UP_NAMES,   "moe up")
    down_added = add_if_present(MLP_DOWN_NAMES, "moe down")
    if MOE_LORA_MODE == "tied":
        print(f"  {'MoE strategy':<18s}: TIED weights across 128 experts (memory-frugal)")
    else:
        print(f"  {'MoE strategy':<18s}: FULL per-expert LoRA (high memory)")
else:
    print(f"  {'mlp / MoE':<18s}: SKIPPED (MOE_LORA_MODE='{MOE_LORA_MODE}')")
print()

print(f"Final LoRA target modules: {LORA_TARGET_MODULES}")

assert LORA_TARGET_MODULES, (
    f"No LoRA target modules detected!\nAll: {dict(linear_suffixes)}"
)

print("Creating trainable LoRA wrapper via FastLanguageModel.get_peft_model ...")
print(f"Unsloth MoE backend: {os.environ.get('UNSLOTH_MOE_BACKEND', 'auto')} (Split LoRA MoE kernels enabled by Unsloth when supported)")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
# ── RESUME from epoch 1 adapter ────────────────────────────────
# RESUME_ADAPTER = "/kaggle/input/datasets/asharamkanderiwal/nemo-adapters"  # path to your dataset
# if os.path.isdir(RESUME_ADAPTER) and os.path.exists(
#     os.path.join(RESUME_ADAPTER, "adapter_model.safetensors")
# ):
#     # peft's load_adapter wires the saved weights into the live LoRA modules
#     # (replacing the freshly-initialized "default" adapter weights in place).
#     model.load_adapter(
#         RESUME_ADAPTER,
#         adapter_name="default",
#         is_trainable=True,
#     )
#     model.set_adapter("default")
#     print(f"[ok] loaded epoch 1 adapter via model.load_adapter from {RESUME_ADAPTER}")

#     # ── SANITY CHECK ───────────────────────────────────────────────
#     # PEFT inits lora_B to ZERO at fresh init. After loading a trained
#     # adapter, lora_B values must be nonzero. If this prints ~0.000, the
#     # load silently failed and you should STOP before training starts.
#     import torch as _t
#     found_b = False
#     for name, p in model.named_parameters():
#         if "lora_B" in name and "default" in name:
#             nrm = p.detach().abs().mean().item()
#             mx  = p.detach().abs().max().item()
#             print(f"     [sanity] {name[:90]}")
#             print(f"              mean|w|={nrm:.6f}  max|w|={mx:.6f}")
#             if nrm < 1e-6:
#                 raise RuntimeError(
#                     "lora_B is at fresh-init zero — adapter load FAILED. "
#                     "STOP and check RESUME_ADAPTER path / file contents."
#                 )
#             found_b = True
#             break
#     assert found_b, "no lora_B params found — model isn't LoRA-wrapped?"
#     print("     [sanity] adapter weights are nonzero ✓ — resume succeeded")
# else:
#     print(f"[info] RESUME_ADAPTER not found at {RESUME_ADAPTER} — fresh init")
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# ============================================================
# fp32 LoRA + bf16 base + fp32 MoE router (numerical hygiene)
# ============================================================
# 0.85 does this. We add an explicit verification.
print("\nCasting LoRA params -> fp32, verifying base/router dtypes...")
n_lora_fp32 = 0
n_router_fp32 = 0
n_base_bf16 = 0
n_other = 0
for name, param in model.named_parameters():
    if ".lora_" in name:
        param.data = param.data.to(torch.float32)
        n_lora_fp32 += 1
    elif ".mixer.gate." in name:
        # NemotronH router weight + e_score_correction_bias are fp32 by design
        if param.dtype == torch.float32:
            n_router_fp32 += 1
        else:
            print(f"  [warn] router param not fp32: {name} -> {param.dtype}")
    else:
        if param.dtype == torch.bfloat16:
            n_base_bf16 += 1
        else:
            n_other += 1
print(f"  LoRA params (fp32)   : {n_lora_fp32}")
print(f"  Router params (fp32) : {n_router_fp32}")
print(f"  Base params (bf16)   : {n_base_bf16}")
print(f"  Other dtypes         : {n_other}  (should be 0 in bf16 mode)")

# ============================================================
# MoE WEIGHT TYING — keeps all 128 expert LoRAs identical
# ============================================================
# Conceptual setup:
#   - Each MoE expert has its own up_proj/down_proj LoRA (PEFT default)
#   - We MAKE THEM IDENTICAL by averaging/broadcasting per training step
#   - Adam stays in sync because gradients are summed, not averaged
#
# Why TIE A on up_proj and B on down_proj?
#   - up_proj.lora_A:  [r, hidden_dim]   — input projection
#   - up_proj.lora_B:  [intermediate, r] — per-expert (NOT tied)
#   - down_proj.lora_A:[r, intermediate] — per-expert (NOT tied)
#   - down_proj.lora_B:[hidden_dim, r]   — output projection
#   The "shared" side touches hidden_dim; tying it keeps expert specialization
#   in the intermediate dim while sharing the hidden-dim adaptation.

moe_tied_params = []
if MOE_LORA_MODE == "tied":
    w1_proj_names = ("gate_up_proj", "up_proj", "gate_proj", ".w1.")
    w2_proj_names = ("down_proj", ".w2.")
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if ".experts." not in name or ".lora_" not in name:
            continue
        is_w1 = any(p in name for p in w1_proj_names)
        is_w2 = any(p in name for p in w2_proj_names)
        is_A = ".lora_A." in name
        is_B = ".lora_B." in name
        should_tie = (is_w1 and is_A) or (is_w2 and is_B)
        if not should_tie:
            continue
        if param.dim() < 2 or param.shape[0] <= 1:
            continue
        moe_tied_params.append(param)

    def _tie_param_init():
        """Make all expert slices identical at start (mean-and-broadcast)."""
        with torch.no_grad():
            for p in moe_tied_params:
                mean = p.data.mean(dim=0, keepdim=True)
                p.data.copy_(mean.expand_as(p.data))

    def _tie_grads():
        """Sum gradients across expert dim, broadcast back. Called pre-step."""
        with torch.no_grad():
            for p in moe_tied_params:
                if p.grad is None:
                    continue
                grad_sum = p.grad.sum(dim=0, keepdim=True)
                p.grad.copy_(grad_sum.expand_as(p.grad))

    print(f"\n[MoE TYING] Identified {len(moe_tied_params)} params to tie")
    if moe_tied_params:
        print(f"  example shapes: {[tuple(p.shape) for p in moe_tied_params[:3]]}")
    _tie_param_init()
    print("  Initial state: all expert slices set to per-param mean (TIED)")
else:
    def _tie_grads():
        pass

# ============================================================
# Trainable parameter audit
# ============================================================
model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable params: {trainable/1e6:.1f}M")

if MOE_LORA_MODE == "tied":
    # Effective trainable = per-expert tied slices count as 1
    expert_param_count = sum(p[0:1].numel() for p in moe_tied_params)
    raw_expert_param_count = sum(p.numel() for p in moe_tied_params)
    effective = trainable - (raw_expert_param_count - expert_param_count)
    print(f"  Effective (tied)   : {effective/1e6:.1f}M  (saves {(raw_expert_param_count - expert_param_count)/1e6:.1f}M from tying)")

if trainable > 900e6:
    print(f"  [warn] >900M trainable — likely high memory; consider MOE_LORA_MODE='exclude' or 'tied'")
elif trainable < 5e6:
    print("  [warn] <5M trainable — too few targets matched")
else:
    print(f"  [ok] healthy LoRA size")

vram_after_lora = torch.cuda.memory_allocated() / 1e9
print(f"\nVRAM after LoRA: {vram_after_lora:.2f} GB")
print(f"VRAM headroom   : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved())/1e9:.2f} GB")

try:
    import triton.backends.nvidia.compiler as nv_compiler
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
    nv_compiler.get_ptxas_version = lambda arch: "12.0"
except Exception as e:
    print(f"[warn] Triton compiler fix skipped: {e}")


In [12]:
# ============================================================
# 11. TRAINING — Stratified SFT  +  CCE loss  +  plain AdamW
# ============================================================
# All v77 fixes integrated:
#   - CCE forward patch          → ~17 GB saved (no logits materialization)
#   - torch.optim.AdamW          → Blackwell-stable (no UVM crash)
#   - max_grad_norm=1.0          → fp32 LoRA tolerates standard clip
#   - save_steps=200 + epoch zip → resume across sessions
#   - NaN guard + Tied-MoE callbacks → from Cell 6
#   - compute_loss override      → bypass TRL entropy_from_logits crash on None

from torch.utils.data import Sampler
import random as _random

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM

# ---- RESUME ACROSS SESSIONS ----
RESUME_FROM_CHECKPOINT = None

# ============================================================
# CCE FORWARD PATCH — no logits tensor ever materialized
# ============================================================
if USE_CCE:
    try:
        from cut_cross_entropy import linear_cross_entropy
        print("[ok] cut_cross_entropy imported")

        _base = model
        while hasattr(_base, "model"):
            _base = _base.model

        if not (hasattr(_base, "backbone") and hasattr(_base, "lm_head")):
            print(f"[warn] CCE patch skipped: model layout missing .backbone/.lm_head")
            print(f"       _base type: {type(_base).__name__}")
            USE_CCE = False
        else:
            _lm_head_module = _base.lm_head
            _orig_forward   = _base.forward

            def _cce_forward(input_ids=None, attention_mask=None, labels=None, **kwargs):
                backbone_out = _base.backbone(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    **{k: v for k, v in kwargs.items()
                       if k in ("position_ids", "past_key_values", "use_cache",
                                "inputs_embeds", "cache_position")},
                )
                hidden_states = backbone_out[0]

                if hasattr(_lm_head_module, "base_layer"):
                    base_w = _lm_head_module.base_layer.weight
                    if hasattr(_lm_head_module, "lora_A") and "default" in _lm_head_module.lora_A:
                        lora_A = _lm_head_module.lora_A["default"].weight
                        lora_B = _lm_head_module.lora_B["default"].weight
                        scaling = _lm_head_module.scaling["default"]
                        lm_weight = base_w + scaling * (lora_B @ lora_A)
                    else:
                        lm_weight = base_w
                else:
                    lm_weight = _lm_head_module.weight

                if labels is not None:
                    shift_hidden = hidden_states[..., :-1, :].contiguous()
                    shift_labels = labels[..., 1:].contiguous()
                    valid = shift_labels != -100
                    if valid.any():
                        loss = linear_cross_entropy(
                            shift_hidden,
                            lm_weight,
                            shift_labels.masked_fill(~valid, 0),
                            reduction="none",
                        )
                        loss = (loss * valid.float()).sum() / valid.float().sum().clamp(min=1)
                    else:
                        loss = shift_hidden.sum() * 0.0
                else:
                    loss = None

                from transformers.modeling_outputs import CausalLMOutputWithPast
                return CausalLMOutputWithPast(
                    loss=loss,
                    logits=None,  # TRL's compute_loss is overridden below to handle this
                    past_key_values=getattr(backbone_out, "past_key_values", None),
                    hidden_states=getattr(backbone_out, "hidden_states", None),
                    attentions=getattr(backbone_out, "attentions", None),
                )

            _base.forward = _cce_forward
            print("[OK] CCE forward patched ✓ (~17GB saved, no logits materialization)")
    except ImportError as e:
        print(f"[warn] CCE not available: {e}")
        USE_CCE = False
    except Exception as e:
        print(f"[warn] CCE patch failed: {e}")
        import traceback; traceback.print_exc()
        USE_CCE = False
else:
    print("[info] CCE disabled — using standard CrossEntropy (uses ~17GB more VRAM)")

# ============================================================
# Stratified sampler — coherent gradient per category (OUR EDGE)
# ============================================================
def build_stratified_index_order(labels, chunk_size, seed=0):
    buckets = {}
    for i, lbl in enumerate(labels):
        buckets.setdefault(lbl, []).append(i)
    rng = _random.Random(seed)
    for lbl in buckets:
        rng.shuffle(buckets[lbl])
    order = []
    active = list(buckets.keys())
    rng.shuffle(active)
    while active:
        next_active = []
        for lbl in active:
            take = buckets[lbl][:chunk_size]
            buckets[lbl] = buckets[lbl][chunk_size:]
            order.extend(take)
            if buckets[lbl]:
                next_active.append(lbl)
        active = next_active
    return order


class PrecomputedOrderSampler(Sampler):
    def __init__(self, labels, chunk_size, num_epochs, base_seed=1337):
        self.labels      = list(labels)
        self.chunk_size  = chunk_size
        self.num_epochs  = num_epochs
        self.base_seed   = base_seed
        self.epoch       = 0
        self._current_order = build_stratified_index_order(
            self.labels, chunk_size, seed=base_seed
        )
    def set_epoch(self, epoch):
        self.epoch = epoch
        self._current_order = build_stratified_index_order(
            self.labels, self.chunk_size, seed=self.base_seed + epoch
        )
    def __iter__(self):
        return iter(self._current_order)
    def __len__(self):
        return len(self.labels)


# ============================================================
# CRITICAL FIX: compute_loss override — bypass TRL entropy_from_logits(None) crash
# ============================================================
# TRL >=0.11 unconditionally calls entropy_from_logits(outputs.logits) inside
# compute_loss when liger_kernel is off. Our CCE returns logits=None to save
# 17 GB → that path crashes with "NoneType has no attribute 'shape'".
#
# Fix: override compute_loss to:
#   1) Run the model (CCE patch already returned loss)
#   2) Return outputs.loss directly without touching outputs.logits

def _cce_compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
    """Compute loss without touching outputs.logits (which is None when CCE is on)."""
    outputs = model(**inputs)
    if outputs.loss is not None:
        loss = outputs.loss
    else:
        loss = torch.zeros((), device=next(model.parameters()).device, requires_grad=True)
    return (loss, outputs) if return_outputs else loss


class StratifiedSFTTrainer(SFTTrainer):
    def __init__(self, *args, stratified_labels=None, chunk_size=16,
                 num_epochs=3, **kwargs):
        self._strat_labels = stratified_labels
        self._strat_chunk  = chunk_size
        self._strat_epochs = num_epochs
        super().__init__(*args, **kwargs)

    def _get_train_sampler(self, *args, **kwargs):
        if self._strat_labels is None:
            return super()._get_train_sampler(*args, **kwargs)
        return PrecomputedOrderSampler(
            labels=self._strat_labels,
            chunk_size=self._strat_chunk,
            num_epochs=self._strat_epochs,
        )

    # CCE-safe loss computation (skips TRL's entropy_from_logits)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        return _cce_compute_loss(self, model, inputs, return_outputs, num_items_in_batch)

    # Plain torch.optim.AdamW (Blackwell-stable, no paged_adamw_8bit UVM crash)
    def create_optimizer(self):
        if self.optimizer is None:
            decay_params = [p for p in self.model.parameters() if p.requires_grad]
            self.optimizer = torch.optim.AdamW(
                decay_params,
                lr=self.args.learning_rate,
                betas=(0.9, 0.95),
                eps=1e-8,
                weight_decay=0.0,
            )
            print(f"[ok] Optimizer: torch.optim.AdamW on {sum(p.numel() for p in decay_params)/1e6:.1f}M params (Blackwell-stable)")
        return self.optimizer


class _PlainTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        return _cce_compute_loss(self, model, inputs, return_outputs, num_items_in_batch)
    def create_optimizer(self):
        if self.optimizer is None:
            decay_params = [p for p in self.model.parameters() if p.requires_grad]
            self.optimizer = torch.optim.AdamW(
                decay_params, lr=self.args.learning_rate,
                betas=(0.9, 0.95), eps=1e-8, weight_decay=0.0,
            )
        return self.optimizer


# ============================================================
# SFTConfig — bf16, plain AdamW, step + epoch checkpoints
# ============================================================
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    save_strategy="steps",
    save_steps=SAVE_EVERY_N_STEPS,
    save_total_limit=1,             # was 3 — disk-full crash on /kaggle/working
    save_only_model=True,           # don't save optimizer state (~5 GB each save)
    report_to="wandb" if WANDB_AVAILABLE else "none",
    run_name=WANDB_RUN_NAME if WANDB_AVAILABLE else None,
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=USE_PACKING,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=2,
    remove_unused_columns=False,
)

labels_for_sampler = list(hf_dataset['label']) if USE_STRATIFIED_BATCHING else None

callbacks = [LiveProgressCallback(), nan_callback, ckpt_callback]
if MOE_LORA_MODE == "tied":
    callbacks.append(tied_callback)
    print("[info] TiedMoEGrad callback active (gradients summed across experts)")

if USE_STRATIFIED_BATCHING:
    print(f"Using STRATIFIED batching (chunk = {EFFECTIVE_BATCH} samples/category)")
    trainer = StratifiedSFTTrainer(
        model=model,
        train_dataset=hf_dataset,
        processing_class=tokenizer,
        args=training_args,
        callbacks=callbacks,
        stratified_labels=labels_for_sampler,
        chunk_size=EFFECTIVE_BATCH,
        num_epochs=NUM_EPOCHS,
    )
else:
    print("Using standard RANDOM batching")
    trainer = _PlainTrainer(
        model=model,
        train_dataset=hf_dataset,
        processing_class=tokenizer,
        args=training_args,
        callbacks=callbacks,
    )

steps_per_epoch = trainer.state.max_steps // NUM_EPOCHS if hasattr(trainer.state, 'max_steps') else "?"
print("=" * 60)
print(f"  v7.7 Training — surpass-0.85 stack")
print("=" * 60)
print(f"  Samples       : {len(hf_dataset)}  |  epochs={NUM_EPOCHS}  |  max_seq={MAX_SEQ_LEN}")
print(f"  Batch         : {BATCH_SIZE}×{GRAD_ACCUM} = {EFFECTIVE_BATCH} eff   steps/epoch≈{steps_per_epoch}")
print(f"  LR            : {LR:.1e}  warmup={WARMUP_STEPS}  schedule=cosine")
print(f"  LoRA          : r={LORA_RANK}, α={LORA_ALPHA}, MoE={MOE_LORA_MODE}")
print(f"  Loss kernel   : {'CUT-CROSS-ENTROPY ✓' if USE_CCE else 'standard CrossEntropy'}")
print(f"  compute_loss  : OVERRIDDEN (bypasses TRL entropy_from_logits crash on None)")
print(f"  Mamba kernel  : {'FAST PATH ✓' if (nemotron_mod and nemotron_mod.is_fast_path_available) else 'PYTHON SCAN ✗'}")
print(f"  Optimizer     : torch.optim.AdamW (β=0.9,0.95)")
print(f"  Stratified    : {USE_STRATIFIED_BATCHING}")
print(f"  Checkpoints   : every {SAVE_EVERY_N_STEPS} steps + per-epoch zip")
print(f"  NaN guard     : ON (halts after 2 consecutive non-finite losses)")
print(f"  Resume from   : {RESUME_FROM_CHECKPOINT or '(fresh start)'}")
print(f"  W&B           : {'offline' if WANDB_AVAILABLE else 'disabled'}\n")

t0 = time.time()
trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
elapsed_hrs = (time.time() - t0) / 3600
print(f"\nTraining complete! Time: {elapsed_hrs:.2f} hrs")

ckpt_callback.print_summary()

peak_vram = torch.cuda.max_memory_allocated() / 1e9
print(f"\nPeak VRAM during training: {peak_vram:.2f} GB / 95 GB")


[ok] cut_cross_entropy imported
[OK] CCE forward patched ✓ (~17GB saved, no logits materialization)
[info] TiedMoEGrad callback active (gradients summed across experts)
Using standard RANDOM batching


Adding EOS to train dataset:   0%|          | 0/10545 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10545 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10545 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


  v7.7 Training — surpass-0.85 stack
  Samples       : 10545  |  epochs=1  |  max_seq=8192
  Batch         : 2×2 = 4 eff   steps/epoch≈0
  LR            : 2.7e-04  warmup=100  schedule=cosine
  LoRA          : r=32, α=48, MoE=tied
  Loss kernel   : CUT-CROSS-ENTROPY ✓
  compute_loss  : OVERRIDDEN (bypasses TRL entropy_from_logits crash on None)
  Mamba kernel  : FAST PATH ✓
  Optimizer     : torch.optim.AdamW (β=0.9,0.95)
  Stratified    : False
  Checkpoints   : every 1000 steps + per-epoch zip
  NaN guard     : ON (halts after 2 consecutive non-finite losses)
  Resume from   : (fresh start)
  W&B           : offline



Training:   0%|          | 0/2637 [00:00<?, ?step/s]

Step,Training Loss
10,1.429264
20,1.220535
30,0.884208
40,0.894307
50,0.458998
60,0.358179
70,0.231014
80,0.174749
90,0.220756
100,0.157621


wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run



[Epoch 01] ✓ adapter_epoch_01.zip  (764.6 MB)  avg_loss=0.1002

Training complete! Time: 7.53 hrs

  Checkpoint loss summary:
    epoch 01: loss=0.1002 ← best

  Best checkpoint: adapter_epoch_01.zip  (use for submission)

Peak VRAM during training: 86.83 GB / 95 GB


In [13]:
# ============================================================
# 12. SAVE FINAL ADAPTER — plain LoRA  (vLLM compatible)
# ============================================================
trainer.model.save_pretrained(OUTPUT_DIR)

config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
with open(config_path) as f:
    adapter_config = json.load(f)

adapter_config["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"
with open(config_path, "w") as f:
    json.dump(adapter_config, f, indent=2)

print(f"base_model_name_or_path -> {adapter_config['base_model_name_or_path']}")
print(f"peft_type              -> {adapter_config.get('peft_type')}")
print(f"r / alpha              -> {adapter_config.get('r')} / {adapter_config.get('lora_alpha')}")
print(f"use_dora               -> {adapter_config.get('use_dora', False)}  (must be False)")
print(f"use_rslora             -> {adapter_config.get('use_rslora', False)}  (must be False)")

try:
    from safetensors import safe_open
    with safe_open(os.path.join(OUTPUT_DIR, "adapter_model.safetensors"),
                   framework="pt") as f:
        keys  = list(f.keys())
        norms = [f.get_tensor(k).norm().item() for k in keys[:5]]
    print(f"\nAdapter tensors: {len(keys)} parameters")
    print(f"First 5 weight norms: {[f'{n:.4f}' for n in norms]}")
    if all(n < 0.001 for n in norms):
        print("WARNING: Norms near 0 — adapter may be untrained!")
    else:
        print("Adapter looks healthy (non-zero weights).")
except Exception as e:
    print(f"Could not verify safetensors: {e}")

print(f"\nFiles in {OUTPUT_DIR}:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"  {fname}  ({size_mb:.2f} MB)")

base_model_name_or_path -> metric/nemotron-3-nano-30b-a3b-bf16
peft_type              -> LORA
r / alpha              -> 32 / 48
use_dora               -> False  (must be False)
use_rslora             -> False  (must be False)

Adapter tensors: 12008 parameters
First 5 weight norms: ['4.0140', '4.4833', '3.2606', '0.0000', '3.4938']
Adapter looks healthy (non-zero weights).

Files in /kaggle/working/adapter:
  README.md  (0.00 MB)
  adapter_config.json  (0.00 MB)
  adapter_model.safetensors  (3373.43 MB)


In [14]:
# ============================================================
# 13. ZIP FINAL ADAPTER + LIST PER-EPOCH CHECKPOINTS
# ============================================================
ZIP_PATH = "/kaggle/working/adapter.zip"
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

file_count = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, arcname=fname)
            file_count += 1

with zipfile.ZipFile(ZIP_PATH) as zf:
    contents = zf.namelist()

zip_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024

print("=" * 60)
print(f"  FINAL adapter.zip  ({zip_mb:.1f} MB)  — {file_count} files")
print(f"  Contents: {contents}")
print("=" * 60)

ckpt_zips = sorted(
    [f for f in os.listdir(CKPT_DIR) if f.endswith(".zip")],
    key=lambda x: int(x.replace("adapter_epoch_", "").replace(".zip", ""))
    if x.replace("adapter_epoch_", "").replace(".zip", "").isdigit() else 0
)

print(f"\nPer-epoch checkpoints in {CKPT_DIR}:")
total_ckpt_mb = 0
for zname in ckpt_zips:
    zpath = os.path.join(CKPT_DIR, zname)
    mb = os.path.getsize(zpath) / 1024 / 1024
    total_ckpt_mb += mb
    epoch_num = zname.replace("adapter_epoch_", "").replace(".zip", "")
    loss = ckpt_callback.epoch_losses.get(int(epoch_num), float("nan")) if epoch_num.isdigit() else float("nan")
    print(f"  {zname:<30}  {mb:6.1f} MB  loss={loss:.4f}")

print(f"\nTotal checkpoint storage: {total_ckpt_mb:.1f} MB  ({len(ckpt_zips)} files)")
print("\nTip: Use the checkpoint with the LOWEST loss for your submission.")

assert "adapter_config.json" in contents, "MISSING adapter_config.json!"
assert "adapter_model.safetensors" in contents, "MISSING adapter_model.safetensors!"

  FINAL adapter.zip  (764.6 MB)  — 3 files
  Contents: ['README.md', 'adapter_config.json', 'adapter_model.safetensors']

Per-epoch checkpoints in /kaggle/working/checkpoints:
  adapter_epoch_01.zip             764.6 MB  loss=0.1002

Total checkpoint storage: 764.6 MB  (1 files)

Tip: Use the checkpoint with the LOWEST loss for your submission.


In [15]:
# ============================================================
# 14. FINAL VERIFICATION — eval-server compliance + memory report
# ============================================================
print("=" * 60)
print("  v7.6 TRAINING SUMMARY")
print("=" * 60)

with open(os.path.join(OUTPUT_DIR, "adapter_config.json")) as f:
    final_cfg = json.load(f)

mode_str = "Hybrid, 4-bit NF4 base (QLoRA)" if USE_QLORA else "Hybrid, bf16 base"
print(f"\n  Model             : Nemotron-3-Nano-30B-A3B ({mode_str})")
print(f"  base_model_name   : {final_cfg.get('base_model_name_or_path')}")
print(f"  LoRA rank (r)     : {final_cfg.get('r')}")
print(f"  LoRA alpha        : {final_cfg.get('lora_alpha')}")
print(f"  Target modules    : {final_cfg.get('target_modules')}")
print(f"  Dropout           : {final_cfg.get('lora_dropout')}")
print(f"  use_dora          : {final_cfg.get('use_dora', False)}")
print(f"  use_rslora        : {final_cfg.get('use_rslora', False)}")
print(f"\n  Dataset           : {len(hf_dataset)} examples (after filtering)")
print(f"  Categories        : {len(set(hf_dataset['label']))}")
print(f"  Epochs            : {NUM_EPOCHS}")
print(f"  Learning rate     : {LR}")
print(f"  Effective batch   : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Training time     : {elapsed_hrs:.2f} hrs")
print(f"  Peak VRAM         : {peak_vram:.2f} GB")
print(f"\n  Adapter zip       : {ZIP_PATH} ({zip_mb:.1f} MB)")
print(f"  Adapter files     : {contents}")

targets = final_cfg.get('target_modules', [])
has_mamba = 'in_proj' in targets       # Mamba's in_proj contains z-gate
has_attn  = 'q_proj' in targets or 'qkv_proj' in targets
has_mlp_gate = any(m in targets for m in
                   ["gate_proj", "gate_up_proj", "w1",
                    "linear_fc1", "fc1", "gate", "wi_0", "wi"])
checks = [
    ("base_model = metric/...",       final_cfg.get('base_model_name_or_path') == 'metric/nemotron-3-nano-30b-a3b-bf16'),
    ("peft_type = LORA",               final_cfg.get('peft_type') == 'LORA'),
    ("attention IN targets",           has_attn),
    ("mamba in_proj IN targets",       has_mamba),
    ("lm_head NOT in targets",         'lm_head' not in targets),
    ("embed_tokens NOT in targets",    'embed_tokens' not in targets),
    ("dropout = 0",                    final_cfg.get('lora_dropout', -1) == 0.0),
    ("rank = 32",                      final_cfg.get('r') == 32),
    ("alpha = 64",                     final_cfg.get('lora_alpha') == 64),
    ("use_dora = False/absent",        not final_cfg.get('use_dora', False)),
    ("use_rslora = False/absent",      not final_cfg.get('use_rslora', False)),
    ("modules_to_save empty/absent",   not final_cfg.get('modules_to_save')),
]

print(f"\n  Verification checks:")
all_ok = True
for name, passed in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_ok = False
    print(f"    [{status}] {name}")

# Informational, not a FAIL: MLP gate is optional on Nemotron-H
gate_status = "PASS" if has_mlp_gate else "INFO (Mamba in_proj covers gating)"
print(f"    [{gate_status}] explicit MLP gate IN targets")

if all_ok:
    print(f"\n  All required checks passed! Adapter is eval-server compliant.")
    print(f"  -> Download adapter.zip (or best per-epoch checkpoint) from Kaggle Output")
    print(f"  -> Use with nemotron_v75_submission.ipynb (or any plain-LoRA submission)")
else:
    print(f"\n  WARNING: Some checks failed — review before submitting.")

if WANDB_AVAILABLE:
    wandb.log({
        "final/training_hours": elapsed_hrs,
        "final/dataset_size": len(hf_dataset),
        "final/adapter_zip_mb": zip_mb,
        "final/peak_vram_gb": peak_vram,
    })
    wandb.finish()
    wandb_dir = os.path.join(WANDB_DIR, "wandb")
    wandb_zip = "/kaggle/working/wandb_logs.zip"
    if os.path.exists(wandb_dir):
        with zipfile.ZipFile(wandb_zip, "w", zipfile.ZIP_DEFLATED) as zf:
            for root, dirs, files in os.walk(wandb_dir):
                for fname in files:
                    fpath = os.path.join(root, fname)
                    arcname = os.path.relpath(fpath, WANDB_DIR)
                    zf.write(fpath, arcname=arcname)
        wandb_zip_mb = os.path.getsize(wandb_zip) / 1024 / 1024
        print(f"\n  W&B logs zipped: {wandb_zip} ({wandb_zip_mb:.1f} MB)")


wandb: 
wandb: Run history:
wandb: epoch_checkpoint/avg_loss ▁
wandb:    epoch_checkpoint/epoch ▁
wandb:   epoch_checkpoint/zip_mb ▁
wandb:      final/adapter_zip_mb ▁
wandb:        final/dataset_size ▁
wandb:        final/peak_vram_gb ▁
wandb:      final/training_hours ▁
wandb:               train/epoch ▁▁▁▁▂▂▂▂▂▂▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇██
wandb:         train/global_step ▁▁▁▁▂▂▂▂▃▃▃▃▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█████
wandb:           train/grad_norm ▇▆▅▅▃▃▂▃▂▃▄▂▂▂▃▂▂▃▂▂▁▁▁▁▁▃▁▁▁▁▁▃▃▂▂▂█▁▁▁
wandb:                        +2 ...
wandb: 
wandb: Run summary:
wandb: epoch_checkpoint/avg_loss 0.10019
wandb:    epoch_checkpoint/epoch 1
wandb:   epoch_checkpoint/zip_mb 764.5863
wandb:      final/adapter_zip_mb 764.58537
wandb:        final/dataset_size 10545
wandb:        final/peak_vram_gb 86.82802
wandb:      final/training_hours 7.52735
wandb:                total_flos 1.0379590564531855e+19
wandb:               train/epoch 1
wandb:         train/global_step 2637
wandb:                      

  v7.6 TRAINING SUMMARY

  Model             : Nemotron-3-Nano-30B-A3B (Hybrid, bf16 base)
  base_model_name   : metric/nemotron-3-nano-30b-a3b-bf16
  LoRA rank (r)     : 32
  LoRA alpha        : 48
  Target modules    : ['out_proj', 'o_proj', 'in_proj', 'down_proj', 'q_proj', 'v_proj', 'k_proj', 'up_proj']
  Dropout           : 0.0
  use_dora          : False
  use_rslora        : False

  Dataset           : 10545 examples (after filtering)
  Categories        : 9
  Epochs            : 1
  Learning rate     : 0.00027
  Effective batch   : 4
  Training time     : 7.53 hrs
  Peak VRAM         : 86.83 GB

  Adapter zip       : /kaggle/working/adapter.zip (764.6 MB)
  Adapter files     : ['README.md', 'adapter_config.json', 'adapter_model.safetensors']

  Verification checks:
    [PASS] base_model = metric/...
    [PASS] peft_type = LORA
    [PASS] attention IN targets
    [PASS] mamba in_proj IN targets
    [PASS] lm_head NOT in targets
    [PASS] embed_tokens NOT in targets
    [PASS] 